<a href="https://colab.research.google.com/github/rudigerroger/FON-GPT/blob/main/Fon_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fon-GPT

Pose une question en fongbe (écrite ou parlée), l'IA te répond en fongbe.

**Chaîne :** voix fongbe → texte (Omnilingual ASR) → français (NLLB)
→ réponse (openai/gpt-oss-20b, via l'API Groq) → fongbe (NLLB) → voix (MMS-TTS).

**Prérequis :** GPU T4 activé, et un secret `GROQ_API_KEY` dans les « Secrets » de Colab.

**Crédits :** Omnilingual ASR, NLLB et MMS-TTS sont de Meta AI. openai/gpt-oss-20b est de OpenAI, hébergé par Groq.

**Licences :** MMS-TTS et NLLB sont sous licence non commerciale (CC-BY-NC).

**Limites :** la traduction fongbe ↔ français est encore faible, et chaque étape de la chaîne
peut ajouter des erreurs. Les questions sont envoyées à Groq.

### **1- Installation**

In [ ]:
# L'écoute tourne dans un environnement Python 3.11 séparé : son moteur n'est pas
# compatible avec la version de Python de Colab.
!apt-get install -y -q libsndfile1 ffmpeg
!pip install -q uv
!uv venv /content/asr_env --python 3.11
!uv pip install --python /content/asr_env/bin/python torch==2.8.0 torchaudio==2.8.0 omnilingual-asr==0.2.0

# Bibliothèques du notebook principal
!pip install -q "transformers>=4.56" sentencepiece gradio groq

## **2 - Les imports**

In [ ]:
import re                               # pour retirer les emojis ou les caractères bizarres avant de le lire à voix haute.
import subprocess                       # pour exécuter des commandes externes depuis Python
import time                             # pour gérer les délais et mesurer la durée d'exécution

import gradio as gr                     # pour créer l'interface web de test
import numpy as np                      # calcule numérique
import torch                            # pour faire tourner les modèles d'IA et utiliser le gpu
from google.colab import userdata       # pour lit les clés API secrètes stockées dans le Colab
from groq import Groq                   # pour communiquer avec les modèles LLM hébergé sur Groq
from scipy.io import wavfile            # pour lire et enregistrer des fichiers audio au format WAV
#Importe 3 outils de la bibliothèque Transformers (AutoModelForSeq2SeqLM, AutoTokenizer, VitsModel)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, VitsModel

print("Imports terminés.")              # On va afficher un message si tout est bien emporté

## **3 - Les réglages**

In [ ]:
# Modèles
MODELE_ECOUTE = "omniASR_CTC_300M"                      # Le modèle qui transcrire la voix fongbe en texte
MODELE_TRADUCTION = "facebook/nllb-200-distilled-1.3B"  # traduit fongbe en français
MODELE_VOIX = "facebook/mms-tts-fon"                    # Le modèle qui convertir le texte fongbe en voix
MODELE_LLM = "openai/gpt-oss-20b"                       # LLM qui fournir les réponses, via l'API Groq

# Langues (codes NLLB)
FON = "fon_Latn"
FRANCAIS = "fra_Latn"

# Traduction
NB_FAISCEAUX = 5   # plus haut il est meilleur mais plus lent

# Le prompt system
CONSIGNE = (
    "Tu es Fon-GPT, un assistant amical. Réponds en français, en deux phrases moyennes "
    "maximum, avec des mots simples, sans liste ni emoji, ni parenthèse."
    "Écris tous les nombres en toutes lettres "
    "Pour une question de vole, refuse poliment que tu ne peux pas repondre."
)
LLM_MAX_TOKENS = 500
LLM_TEMPERATURE = 0.6
NB_ECHANGES_MEMOIRE = 3   # nombre de questions/réponses dont le bot se souvient

# Voix
VOIX_VITESSE = 0.85       # plus bas = plus lent
VOIX_VARIATION = 0.4      # plus bas = intonation plus calme
VOIX_RYTHME = 0.5         # plus bas = durée des syllabes plus régulière
VOIX_SEED = 555           # fixe le hasard pour retrouver la même voix
PAUSE_ENTRE_PHRASES = 0.3 # Le nombre de secondes de pause entre deux phrases

# Écoute
PYTHON_ECOUTE = "/content/asr_env/bin/python"
DUREE_MAX_AUDIO = 30      # Le nombre de secondes maxi de l'audio

DEVICE = "cuda"

print("Réglages chargés. GPU disponible :", torch.cuda.is_available()) # Afficher un message si tout marche bien

### **4 - Le script d'écoute**

In [ ]:
%%writefile ecoute.py
# Transcrit un fichier audio
import sys
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

audio, modele = sys.argv[1], sys.argv[2]
pipeline = ASRInferencePipeline(model_card=modele)
print(pipeline.transcribe([audio], batch_size=1)[0])

## **5 - Chargement des modèles**

In [ ]:
# Traduction
tok_traduction = AutoTokenizer.from_pretrained(MODELE_TRADUCTION)
modele_traduction = AutoModelForSeq2SeqLM.from_pretrained(
    MODELE_TRADUCTION, dtype=torch.float16
).to(DEVICE)

# Voix
tok_voix = AutoTokenizer.from_pretrained(MODELE_VOIX)
modele_voix = VitsModel.from_pretrained(MODELE_VOIX)
modele_voix.speaking_rate = VOIX_VITESSE
modele_voix.noise_scale = VOIX_VARIATION
modele_voix.noise_scale_duration = VOIX_RYTHME
FREQUENCE = modele_voix.config.sampling_rate

# Cerveau : la clé est lue dans les « Secrets » de Colab (au niveau de l'icone de clé à gauche), ne jamais le mètre écrite dans le code
client_llm = Groq(api_key=userdata.get("GROQ_API_KEY"))

# Afficher un message si tout marche bien
print(f"Modèles chargés. GPU utilisé : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

## **6 - Les fonctions**

In [ ]:
historique: list[dict] = []   # Mémoire de la conversation


def decouper_en_phrases(texte: str) -> list[str]:
    # Coupe un texte en phrases : la traduction et la voix marchent mieux sur du court
    return [p for p in re.split(r"(?<=[.!?])\s+", texte.strip()) if p]


def ecouter(chemin_audio: str) -> str:
    # Transcrit un audio fongbe en texte
    # Format attendu par le modèle : un canal, 16 000 Hz, durée limitée
    subprocess.run(
        ["ffmpeg", "-y", "-i", chemin_audio, "-t", str(DUREE_MAX_AUDIO),
         "-ac", "1", "-ar", "16000", "entree.wav"],
        check=True, capture_output=True,
    )
    resultat = subprocess.run(
        [PYTHON_ECOUTE, "ecoute.py", "entree.wav", MODELE_ECOUTE],
        capture_output=True, text=True,
    )
    if resultat.returncode != 0:
        raise RuntimeError(resultat.stderr[-500:])
    lignes = resultat.stdout.strip().splitlines()
    return lignes[-1] if lignes else ""


@torch.no_grad()
def traduire(texte: str, langue_source: str, langue_cible: str) -> str:
    # Traduit un texte entre le fongbe et le français
    tok_traduction.src_lang = langue_source
    entrees = tok_traduction(texte, return_tensors="pt").to(DEVICE)
    sortie = modele_traduction.generate(
        **entrees,
        forced_bos_token_id=tok_traduction.convert_tokens_to_ids(langue_cible),
        num_beams=NB_FAISCEAUX,
        max_new_tokens=200,
    )
    return tok_traduction.batch_decode(sortie, skip_special_tokens=True)[0]


def penser(question_fr: str) -> str:
    #Envoie la question au LLM (avec les derniers échanges) et renvoie sa réponse
    historique.append({"role": "user", "content": question_fr})
    messages = [{"role": "system", "content": CONSIGNE}]
    messages += historique[-2 * NB_ECHANGES_MEMOIRE:]
    reponse = client_llm.chat.completions.create(
        model=MODELE_LLM, messages=messages,
        max_tokens=LLM_MAX_TOKENS, temperature=LLM_TEMPERATURE,
        reasoning_effort="low",   # réflexion courte : réponse plus rapide
    ).choices[0].message.content or ""
    reponse = reponse.strip()
    historique.append({"role": "assistant", "content": reponse})
    return reponse


@torch.no_grad()
def parler(texte_fon: str) -> str | None:
    # Lit un texte fongbe phrase par phrase et renvoie le chemin du fichier wav
    silence = np.zeros(int(PAUSE_ENTRE_PHRASES * FREQUENCE), dtype=np.float32)
    morceaux = []
    for phrase in decouper_en_phrases(texte_fon):
        entrees = tok_voix(phrase.lower(), return_tensors="pt")
        if entrees["input_ids"].shape[1] == 0:   # Si aucune lettre reconnue
            continue
        torch.manual_seed(VOIX_SEED)
        morceaux += [modele_voix(**entrees).waveform[0].numpy(), silence]
    if not morceaux:
        return None
    fichier = f"reponse_{int(time.time())}.wav"
    wavfile.write(fichier, FREQUENCE, np.concatenate(morceaux))
    return fichier


def repondre(audio, texte_fon, texte_fr):
    # Chaîne complète. Priorité des entrées : français écrit > fongbe écrit > audio
    texte_fon = (texte_fon or "").strip()
    texte_fr = (texte_fr or "").strip()

    if not texte_fr:
        texte_fon = texte_fon or (ecouter(audio) if audio else "")
        if not texte_fon:
            return "", "", "", "", None
        texte_fr = traduire(texte_fon, FON, FRANCAIS)

    reponse_fr = penser(texte_fr)
    reponse_fon = " ".join(
        traduire(phrase, FRANCAIS, FON) for phrase in decouper_en_phrases(reponse_fr)
    )
    return texte_fon, texte_fr, reponse_fr, reponse_fon, parler(reponse_fon)


print("Fonctions prêtes.")              # Afficher un message si tout marche bien

## **7 - L'interface**

In [ ]:
# Ici l'interface de test fait avec gradio
# Parfois, le micro ne marche pas bien sur colab, cliquer sur le public URL, l'interface s'ouvrira dans votre navigateur

with gr.Blocks(title="Fon-GPT") as demo:
    texte_fon = gr.Textbox(label="Question en fongbe (écrite)")
    micro = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Parle ou envoie un audio")

    envoyer = gr.Button("Envoyer", variant="primary")
    nouvelle = gr.Button("Nouvelle conversation")

    sorties = [
        gr.Textbox(label=nom)
        for nom in ["Fongbe", "Compris en français", "Réponse en français", "Réponse en fongbe"]
    ]
    lecteur = gr.Audio(label="Réponse parlée", autoplay=True)

    envoyer.click(repondre, [micro, texte_fon], sorties + [lecteur])
    nouvelle.click(historique.clear)

demo.launch(share=True, debug=True)